# Unit 05 - Strategies and Challenges in Federated Learning

**Authors**:

- Patricia Guadalupe Alvarenga Mairena
- Laura González Lemos
- Iria Janeiro Pazos
- Ayesha Munir 
- Andrea Real Blanco

# Strategies and Challenges in Federated Learning

In the previous unit, we introduced the fundamentals of **Federated Learning** and implemented a complete end-to-end federated training pipeline using Flower. We focused on understanding the interaction between clients and server, the role of local training, and the basic Federated Averaging (FedAvg) algorithm.

In this unit, we move one step further. Rather than treating federated learning as a single algorithm, we explore it as a **family of strategies** designed to address practical challenges that arise in real-world deployments. These challenges include data heterogeneity across clients, limited communication budgets, partial client participation, and unstable convergence.

The main objective of this notebook is twofold:
1. To introduce and conceptually understand **alternative federated learning strategies beyond FedAvg**.
2. To analyze how these strategies relate to specific challenges and how they can be explored experimentally.

Throughout the notebook, we will deliberately reuse components from the previous unit. Some code fragments are intentionally left incomplete as short exercises, allowing you to refresh key concepts and actively engage with the design of federated learning systems.

## Recap Exercise: Rebuilding the Federated Learning Baseline (from Unit 04)

Before introducing new strategies, we briefly revisit the baseline federated learning pipeline implemented in the previous unit. The goal is to refresh the key components required to run a federated learning experiment in Flower:

- a client implementing the `NumPyClient` interface,
- a `ClientApp` that instantiates one client per partition,
- a `ServerApp` that defines the strategy and the number of rounds,
- and the simulation entrypoint.

Complete the following skeleton by filling in the missing pieces.

> **Important:** This code is intentionally incomplete. Do **not** run it as-is. Create a new code cell, copy the completed version there, and execute it.

In [ ]:
import flwr as fl  
import tensorflow as tf
from flwr.common import Context, Metrics
from flwr.server import ServerApp, ServerAppComponents
from flwr.simulation import run_simulation
import numpy as np
from typing import List, Tuple, Dict
from flwr.clientapp import ClientApp

import flwr as fl


NUM_CLIENTS = 10
RANDOM_SEED = 42

num_rounds = 3

#load CIFAR problem and partition it
def unison_shuffled_copies(a: np.ndarray, b: np.ndarray, seed: int = RANDOM_SEED):
    """Shuffle two arrays in unison, preserving alignment between inputs and labels."""
    assert len(a) == len(b)
    rng = np.random.default_rng(seed)
    p = rng.permutation(len(a))
    return a[p], b[p]

def split_index(a: np.ndarray, n: int):
    """Return a list of index arrays splitting 'a' into 'n' approximately equal parts."""
    return np.array_split(np.arange(len(a)), n)

def load_datasets(num_clients: int, train_size: int = 10_000, test_size: int = 1_000):
    """
    Load CIFAR-10, normalize, shuffle, and split it into per-client (train, val, test) tuples.

    Returns:
        train_splits: list of (x_train_i, y_train_i) for each client
        val_splits:   list of (x_val_i,   y_val_i)   for each client
        test_splits:  list of (x_test_i,  y_test_i)  for each client
    """
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    # Normalize to [0, 1]
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    # Keep the notebook lightweight (faster for classroom execution)
    x_train, y_train = x_train[:train_size], y_train[:train_size]
    x_test, y_test = x_test[:test_size], y_test[:test_size]

    # Flatten labels from shape (N, 1) to (N,) to simplify downstream handling
    y_train = y_train.squeeze()
    y_test = y_test.squeeze()

    # Shuffle (fixed seed for reproducibility in class)
    x_train, y_train = unison_shuffled_copies(x_train, y_train, seed=RANDOM_SEED)
    x_test, y_test = unison_shuffled_copies(x_test, y_test, seed=RANDOM_SEED + 1)

    # Split indices per client
    train_index = split_index(x_train, num_clients)
    test_index = split_index(x_test, num_clients)

    train_splits, val_splits, test_splits = [], [], []
    
    for cid in range(num_clients):
        client_train_idx = train_index[cid]
        client_test_idx = test_index[cid]

        # Per-client train split
        x_c, y_c = x_train[client_train_idx], y_train[client_train_idx]

        # 10% validation (kept simple and deterministic)
        val_size = max(1, len(x_c) // 10)
        x_val, y_val = x_c[:val_size], y_c[:val_size]
        x_tr, y_tr = x_c[val_size:], y_c[val_size:]

        train_splits.append((x_tr, y_tr))
        val_splits.append((x_val, y_val))
        test_splits.append((x_test[client_test_idx], y_test[client_test_idx]))

    return train_splits, val_splits, test_splits
    
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

# Define a simple neural network model using TensorFlow and Keras
def generate_ann():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64,  activation="relu"),
        tf.keras.layers.Dense(10,  activation="softmax"),
        ])
    
    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model

def get_parameters(model) -> List[np.ndarray]:
    """Extract model parameters as a list of NumPy arrays."""
    return model.get_weights()

def set_parameters(model, parameters: List[np.ndarray]):
    """Load parameters into a Keras model."""
    model.set_weights(parameters)
    return model

def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    """Weighted average of client accuracies (by number of examples)."""
    if not metrics:
        return {}
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}
    
### Part A — Client logic
class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data  # (x_train, y_train)
        self.val_data = val_data      # (x_val, y_val)

    def get_parameters(self, config):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        # Update local model with global parameters
        set_parameters(self.model, parameters)

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=1,
            batch_size=32,
            verbose=0,
        )

        # Return updated parameters and number of training examples
        return get_parameters(self.model), len(x_train), {}
    
    def evaluate(self, parameters, config):
        # Update local model with global parameters
        set_parameters(self.model, parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        # Return loss, number of evaluation examples, and metrics
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

### Part B — ClientApp (mapping partitions to clients)
from flwr.common import Context
from flwr.clientapp import ClientApp

def client_fn(context: Context) -> fl.client.Client:
    """Create a Flower client for a given simulated node."""
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)


    # Each client gets its own model instance
    model = generate_ann()

    # Select this client's local data partition
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    # Return a Flower client
    return MyClient(model, train_data, val_data).to_client()
    
client_app = ClientApp(client_fn=client_fn)


### Part C — ServerApp (strategy + config)
from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(get_parameters(model))
    del model  # free memory early
    

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=0.5,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

### Part D — Run the experiment
from flwr.simulation import run_simulation

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated clients
)

C:\Users\iriai\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-18 15:20:05,155	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
C:\Users\iriai\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial globa

### Reflection questions

1. Which parts of this pipeline run on the client side and which run on the server side?

2. What information is exchanged between server and clients at each round?

3. Why is it important to return the correct number of examples from fit and evaluate?

`Answer here`


1. The client side is responsible for all the operations that involve local data and local computation. The client loads its data partition, receives the global model parameters from the server, updates the local model using its own training data inside the fit() method, and evaluates the model on its validation data in the "evaluate()" method. It also provides the model parameters through "get_parameters()" when requested. This means that the training and evaluation are performed locally on each client device.

    The server side coordinates the whole learning process. The server initializes the global model, selects which clients participate in each round, sends the current global parameters to them, and collects their updates. After receiving the client results, it aggregates the parameters using a strategy such as Federated Averaging (FedAvg) to produce a new global model. The server also manages the training configuration and number of rounds.


2. At the beginning of each round, the server sends the current global model parameters to the selected clients so that they can synchronize their local models. After training locally, each client sends back its updated model parameters. During evaluation, clients send the evaluation loss, the number of evaluation samples, and optional metrics such as accuracy. The server then aggregates these updates and metrics to update the global model and compute global performance statistics.



3. Returning the correct number of examples is important because it allows the server to correctly interpret and aggregate the results coming from different clients. In federated learning, clients may have very different datasets sizes, so the server needs to know how much data each update or evaluation result represents. Without this information, it would be difficult to combine the client contributions in a meaningful way.





It may be worth mentioning that Flower, by default, initializes the global model by making a call to one random client before distributing it to the remaining clients. However, sometimes more control is required, such as when performing fine-tuning. In such situations, we use server-side initialization, and the `initial_parameters` parameter will hold the initial version of the model for all clients. It is important to note that this parameter must be a serialization of the data, so the utility function `ndarrays_to_parameters` can be quite handy in this case.

## Federated Learning Strategies (and why custom strategies matter)

Federated learning is not defined by a single training algorithm. Instead, it comprises a **family of strategies** that specify how the server orchestrates training: which clients participate, what instructions they receive, how their updates are combined, and how the global model is updated across rounds.

In Flower, these choices are encapsulated in **server-side strategies** (e.g., `FedAvg`). Importantly, strategies are not limited to the ones provided by the library. In many real deployments, researchers and practitioners implement **custom strategies** to match the requirements of a specific application, such as:
- coping with highly non-IID data,
- handling unreliable or slow clients,
- improving convergence with server-side optimizers,
- enforcing fairness constraints or robust aggregation rules,
- or integrating privacy/security mechanisms.

In the next sections, we first introduce FedAvg as a baseline strategy, and then discuss representative alternatives. Throughout the notebook, keep in mind that each strategy can be interpreted as a different design choice about *how the server should use the information coming from clients*.


## Baseline Strategy: Federated Averaging (FedAvg)

The most widely used baseline in federated learning is **Federated Averaging (FedAvg)**. At a high level, FedAvg alternates between two steps:

1. The server sends the current global model parameters to a subset of clients.
2. Each selected client performs local training on its private data and returns updated parameters to the server.

The server then aggregates the client updates—typically using a **weighted average**, where each client’s contribution is proportional to the number of local training examples. This simple mechanism often works well when client data are reasonably similar, but its performance can degrade under strong heterogeneity (e.g., highly non-IID data or very different client behaviors).

In Flower, FedAvg is implemented as a server-side strategy. The main hyperparameters you can control include the fraction of clients participating per round, the minimum number of available clients, and (optionally) the initialization of the global model parameters. In the next sections, we will use FedAvg as a reference point to motivate and understand more advanced strategies.

This has should have been already experimented in the previous lesson and the recap of this one.

## Beyond FedAvg: Alternative and Custom Federated Strategies

While FedAvg provides a simple and effective baseline, many real-world federated learning scenarios violate its underlying assumptions. In particular, client datasets are often **non-IID**, clients may have very different computational capabilities, and only a subset of clients may be available at any given time. These issues motivate the development of **alternative federated learning strategies**.

Several extensions of FedAvg have been proposed to address these challenges. For example, **FedProx** introduces a regularization term in the local objective to prevent client models from drifting too far from the global model, which can improve stability under data heterogeneity. Other approaches, such as **FedOpt** methods (e.g., FedAdam, FedYogi), modify the server-side update rule by applying adaptive optimization techniques at the aggregation step instead of relying on simple averaging.

An important design principle in Flower is that strategies are **modular and extensible**. Researchers and practitioners are not restricted to predefined strategies: it is possible to implement **custom strategies** by subclassing the strategy interface and redefining how client updates are selected, aggregated, or interpreted. This flexibility is essential in practice, as many applications require domain-specific constraints, robustness mechanisms, or fairness-aware aggregation rules that go beyond standard algorithms.

In the following sections, we will connect these strategies to the main challenges in federated learning and explore how changing the server-side strategy impacts training dynamics and model performance.

## Centralized vs Federated Evaluation

So far, evaluation has been treated as part of the federated learning loop, but it is important to make explicit that **there are multiple ways to evaluate a federated model**, and that this choice is closely tied to the server-side strategy.

Broadly speaking, two evaluation paradigms can be distinguished: **centralized (server-side) evaluation** and **federated (client-side) evaluation**.

In **centralized evaluation**, the server evaluates the aggregated global model on a fixed dataset that is not used for training. This approach closely resembles traditional centralized machine learning. It has the advantage of producing stable and reproducible evaluation results, since the evaluation dataset is always the same. In addition, it avoids extra communication with clients during evaluation rounds. However, centralized evaluation assumes that the server has access to a representative evaluation dataset, which is not always realistic in privacy-sensitive or highly decentralized scenarios.

In **federated evaluation**, evaluation is performed by the clients using their local datasets. The server sends the current global model to the clients, each client evaluates it locally, and the resulting metrics are sent back and aggregated by the server. This approach often better reflects real-world federated deployments, as it relies exclusively on decentralized data. At the same time, it introduces additional challenges: evaluation results may fluctuate across rounds due to partial client participation, changing local datasets, or heterogeneous data distributions. Moreover, federated evaluation increases communication costs, since models must be transmitted to clients for evaluation.

The examples used so far in this notebook rely on **federated evaluation**, as they implement an `evaluate` method on the client side and aggregate the resulting metrics on the server. As we will see next, controlling how evaluation is performed and aggregated is another reason why customizing federated learning strategies is often necessary.


## Implementing Custom Strategies in Flower (Code Skeleton)

In Flower, a federated learning strategy is implemented on the **server side** and defines the overall orchestration of the learning process. While built-in strategies such as FedAvg cover many standard use cases, real-world federated learning systems often require **custom strategies** tailored to specific constraints or objectives.

Implementing a custom strategy typically involves extending an existing one (most commonly `FedAvg`) and overriding selected parts of its behavior. These extensions may affect how client updates are aggregated, how clients are sampled at each round, which configuration parameters are sent to clients (for example, the number of local epochs or steps per epoch), or how evaluation metrics are combined and reported.

The following code provides a minimal **skeleton of a custom FedAvg-like strategy**. It highlights the key extension points without introducing unnecessary complexity.

> **Exercise:** Complete the TODOs in the skeleton below and replace the baseline FedAvg strategy in the `ServerApp` with your custom implementation. Observe how changing server-side logic influences the training dynamics and evaluation results.


**Note**: This strategy template is based on Flower versions prior to 1.2. [Example](https://flower.ai/docs/framework/1.19/en/how-to-aggregate-evaluation-results.html)

In [2]:
from typing import List, Tuple, Optional, Dict, Any
import flwr as fl
from flwr.common import Metrics
from flwr.server.client_proxy import ClientProxy

class MyCustomStrategy(fl.server.strategy.FedAvg):
    """Example of a custom strategy extending FedAvg."""

    def __init__(
        self,
        local_epochs: int = 1,
        steps_per_epoch: Optional[int] = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.local_epochs = local_epochs
        self.steps_per_epoch = steps_per_epoch

    def configure_fit(
        self,
        server_round: int,
        parameters: fl.common.Parameters,
        client_manager: fl.server.client_manager.ClientManager,
    ):
        """Send custom config to clients before local training."""
        fit_ins_list = super().configure_fit(server_round, parameters, client_manager)

        # NOTE: This only has an effect if the client reads these values from `config`
        for client, fit_ins in fit_ins_list:
            # TODO: send the hyperparameters to clients via the config dict
            fit_ins.config["local_epochs"] = self.local_epochs
            # TODO: optionally send steps_per_epoch only when not None
            fit_ins.config["steps_per_epoch"] = self.steps_per_epoch

        return fit_ins_list

    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, fl.common.EvaluateRes]],
        failures,
    ):
        """Aggregate evaluation results (custom metrics aggregation)."""
        if not results:
            return None, {}

        # TODO: compute weighted average accuracy across clients
        # Hint: each EvaluateRes has `.num_examples` and `.metrics["accuracy"]`
        total_examples = sum(res.num_examples for _, res in results)      
        weighted_acc = sum(res.metrics["accuracy"] * res.num_examples for _, res in results)

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)
        
        print(f"[Server] Round {server_round} — aggregated accuracy: {weighted_acc / total_examples:.4f}")
        
        return aggregated_loss, {"accuracy": weighted_acc / total_examples}

## Using the Custom Strategy in the ServerApp

Defining a custom strategy is only the first step. To actually use it in a federated learning experiment, we must instantiate it on the server side and return it from the `ServerApp`.

In the next code cell, replace the baseline `FedAvg` strategy with `MyCustomStrategy`. Then, run the simulation and verify two things:

1. The client receives the hyperparameters you injected through `configure_fit` (via the `config` dictionary).
2. The server reports an aggregated accuracy computed by your custom `aggregate_evaluate` implementation.

**Note:** If you do not modify the client’s `fit` method to read `local_epochs` and `steps_per_epoch` from `config`, changing `configure_fit` will have no effect.


In [3]:
# Example: instantiate and use the custom strategy in the ServerApp
NUM_ROUNDS = 5
def server_fn(context: Context):
    # Initialize global model parameters (recommended)
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    # TODO: Replace FedAvg by MyCustomStrategy and pass meaningful values
    strategy = MyCustomStrategy(
        local_epochs=2,          # e.g., 1 or 2
        steps_per_epoch=5,       # e.g., 1, 3, 5 (or None)
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


# Client-side logic adapted to receive custom configuration from the server

class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data  # (x_train, y_train)
        self.val_data = val_data      # (x_val, y_val)

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        # Load global parameters
        self.model.set_weights(parameters)

        # TODO: read hyperparameters sent by the server
        # Hint: these values are injected by the custom strategy in `configure_fit`
        local_epochs = config.get("local_epochs", 1)
        steps_per_epoch = config.get("steps_per_epoch", None)

        print(f"[Client] Received config — local_epochs: {local_epochs}, steps_per_epoch: {steps_per_epoch}")

        x_train, y_train = self.train_data

        # TODO: train the model using the received configuration
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        # TODO: return updated parameters and number of training examples
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        # Load global parameters
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data

        # Evaluate locally
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        # TODO: return loss, number of evaluation examples, and metrics
        return float(loss), len(x_val), {"accuracy": float(accuracy)}



In [4]:
#TODO execute the simulation
def client_fn(context: Context) -> fl.client.Client:
    cid = int(context.node_config.get("partition-id", context.node_id))
    model = generate_ann()
    return MyClient(model, trainloaders[cid], valloaders[cid]).to_client()

client_app = ClientApp(client_fn=client_fn)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=4056) 2026-03-18 15:22:36.049138: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(pid=gcs_server) [2026-03-18 15:22:38,436 E 30692 30688] (gcs_server.exe) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(

(ClientAppActor pid=2708) [Client] Received config — local_epochs: 2, steps_per_epoch: 5


(ClientAppActor pid=2708) C:\Users\iriai\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\backend\tensorflow\core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
(ClientAppActor pid=2708)   return np.array(x)


(ClientAppActor pid=27160) [Client] Received config — local_epochs: 2, steps_per_epoch: 5 [repeated 8x across cluster]


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 2]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 — aggregated accuracy: 0.1590
(ClientAppActor pid=4056) [Client] Received config — local_epochs: 2, steps_per_epoch: 5 [repeated 2x across cluster]
(ClientAppActor pid=18224) [Client] Received config — local_epochs: 2, steps_per_epoch: 5 [repeated 8x across cluster]


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 113.31s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.301664876937866
INFO :      		round 2: 2.2984541416168214
INFO :      		round 3: 2.2938361167907715
INFO :      		round 4: 2.2785248517990113
INFO :      		round 5: 2.249403142929077
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.12300000004470349),
INFO :      	              (2, 0.11099999994039536),
INFO :      	              (3, 0.15700000002980233),
INFO :      	              (4, 0.15900000035762787),
INFO :      	              (5, 0.15399999916553497)]}
INFO :      


[Server] Round 5 — aggregated accuracy: 0.1540
(ClientAppActor pid=5172) [Client] Received config — local_epochs: 2, steps_per_epoch: 5


(ClientAppActor pid=18224) 2026-03-18 15:22:54.991335: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations. [repeated 7x across cluster]
(ClientAppActor pid=18224) To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags. [repeated 7x across cluster]
(ClientAppActor pid=30648) C:\Users\iriai\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\backend\tensorflow\core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword [repeated 7x across cluster]
(ClientAppActor pid=30648)   return np.array(x) [repeated 7x across 

### Strategy Spotlight: FedProx (Stabilizing Training under Heterogeneity)

A major limitation of FedAvg is that it can become unstable when client data are strongly **non-IID**. In such cases, local updates may drift in very different directions, and simple averaging may slow down convergence or even lead to oscillations.

**FedProx** addresses this issue by modifying the *local* optimization objective. Instead of minimizing only the client’s local loss, FedProx adds a **proximal term** that penalizes large deviations from the current global model. Intuitively, this keeps local training “closer” to the global solution and can improve robustness under heterogeneity. This proximal term is typically controlled by a hyperparameter (often denoted as *μ*), which determines how strongly local updates are constrained.

From a practical perspective, FedProx can be seen as a strategy that changes what happens inside the client’s `fit` step: local training becomes more conservative when client distributions differ substantially. This makes FedProx a good example of an approach where the main “strategy change” is not only server-side aggregation, but also how local optimization is performed.

In the next section, we will simulate heterogeneity and discuss why strategies like FedProx (or custom variants) are often considered when FedAvg underperforms.

#### Advanced Note: Building a Strategy from Scratch

So far, we have customized federated learning behavior by **extending an existing strategy** (e.g., subclassing `FedAvg`). This is often the most practical approach, since it allows you to reuse a well-tested baseline and modify only the parts you need (sampling, configuration, aggregation, metrics).

In some research or engineering scenarios, however, it can be useful to implement a strategy **from scratch** by extending Flower’s strategy interface (`flwr.server.strategy.Strategy`). This gives full control over the server-side workflow, including how clients are selected, which instructions are sent each round, and how results are aggregated into a new global model.

Because implementing a full strategy requires more boilerplate and careful handling of edge cases (failures, partial participation, timeouts), we treat it as an advanced topic and focus on extending existing strategies in this notebook.

For an end-to-end example of a strategy built from scratch, see the official tutorial:
[Build a Strategy from Scratch](https://flower.ai/docs/framework/tutorial-series-build-a-strategy-from-scratch-pytorch.html#Build-a-Strategy-from-scratch)


# Challenges in Federated Learning

While federated learning addresses important limitations of centralized machine learning—such as data privacy and the need to move large datasets—it also introduces a set of **new challenges** that directly affect training dynamics and strategy design. In this section, we focus on one of the most fundamental of these challenges: **non-IID data**.

## Non-IID Data

In many machine learning settings, it is common to assume that data are **independent and identically distributed (IID)**. Under this assumption, each data sample is generated independently and follows the same underlying distribution. This assumption simplifies both theoretical analysis and practical algorithm design.

Federated learning, however, rarely satisfies this assumption. Because data are collected and stored locally on different devices, each client typically observes data drawn from a **different distribution**. This leads to **non-IID data**, where statistical properties vary significantly across clients.


![Diagram with IID and non-IID data](https://datasciences.org/wp-content/themes/dslabNew/images/datasciences/IIDness.png)
Credit: [Source of the image](https://datasciences.org/non-iid-learning/)


Non-IID data can arise for many reasons. Data may be correlated over time, influenced by user behavior, or biased toward specific classes or patterns. For example, one client may predominantly collect images of cats, while another may mostly contain images of dogs. As a result, local models trained on different clients may move in very different directions during optimization.

This heterogeneity poses a significant challenge for federated learning. Simple aggregation strategies such as FedAvg implicitly assume that local updates are roughly aligned. When this assumption is violated, training may become unstable, convergence may slow down, or the global model may oscillate between incompatible solutions.

From a strategy-design perspective, non-IID data is one of the main motivations for **alternative and custom federated learning strategies**. Approaches such as FedProx aim to constrain local updates, while custom aggregation rules may reweight or filter client contributions to reduce the impact of extreme heterogeneity. In Flower, addressing non-IID data typically involves modifying the **server-side strategy**, either by extending existing strategies or by implementing custom aggregation logic.

In the next section, we will see how such ideas can be translated into concrete strategy implementations.


## System and Device Heterogeneity

Beyond data heterogeneity, federated learning systems must also cope with **heterogeneity in hardware, software, and connectivity** across clients. Unlike centralized settings—where training typically runs on homogeneous clusters—federated learning involves devices with very different computational capabilities, memory limits, energy constraints, and network conditions.

In practice, this means that some clients may train much faster than others, some may only be intermittently available, and others may fail to complete local training altogether. These effects are often referred to as **system heterogeneity** and can significantly influence both efficiency and convergence.

From a learning perspective, system heterogeneity interacts with strategy design in several ways. Limiting the number of local epochs, adjusting `steps_per_epoch`, or sampling only a subset of clients per round are common techniques to reduce stragglers and stabilize training. From a systems perspective, strategies may also need to tolerate partial participation and client dropouts without compromising robustness.

In Flower, these issues are typically addressed at the **strategy level**, by controlling client sampling, configuring local workloads through the `config` dictionary, and deciding how to handle missing or failed client updates. In the following section, we will examine how communication constraints further shape these design choices.


In [9]:
import random
class HeterogeneousClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Simulate system heterogeneity via different local computation budgets
        # TODO: change the rules according to the exercise (e.g., make 1 client extremely slow)
        #Hit: You can use self.cid to select a specific client
        if random.random()>0.5:
            # "Fast" clients (more compute)
            local_epochs = 5
            steps_per_epoch = 5
        else:
            # "Slow" clients (less compute)
            local_epochs = 3
            steps_per_epoch = 3

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}
        

def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return HeterogeneousClient(cid, model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)

In [10]:
from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]

        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}


from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model and create initial global parameters
    # TODO: create the model and extract initial parameters
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = AggregateCustomMetricStrategy(
        # TODO: set basic FedAvg parameters
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    # Define ServerConfig
    # TODO: set the number of rounds
    config = fl.server.ServerConfig(num_rounds=5)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


from flwr.simulation import run_simulation

NUM_CLIENTS =10
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=gcs_server) [2026-03-18 15:04:25,887 E 39812 43712] (gcs_server.exe) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2026-03-18 15:04:32,750 E 10848 37380] (raylet.exe) main.cc:1032: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(pid=12212) 2026-03-18 15:04:52.3659

[Server] Round 1 aggregated accuracy: 0.1400


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 2 aggregated accuracy: 0.1060


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.1530


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 195.95s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.3055005788803102
INFO :      		round 2: 2.295239567756653
INFO :      		round 3: 2.282360649108887
INFO :      		round 4: 2.2520530462265014
INFO :      		round 5: 2.2267182588577272
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.13999999910593033),
INFO :      	              (2, 0.10599999986588955),
INFO :      	              (3, 0.1449999988079071),
INFO :      	              (4, 0.15299999937415124),
INFO :      	              (5, 0.16999999955296516)]}
INFO :      


[Server] Round 5 aggregated accuracy: 0.1700


(ClientAppActor pid=12212) WARNING :   Manually terminating ClientAppActor
(ClientAppActor pid=45888) C:\Users\iriai\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\backend\tensorflow\core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword [repeated 7x across cluster]
(ClientAppActor pid=45888)   return np.array(x) [repeated 7x across cluster]


## Exercise: Exploring the Effect of System Heterogeneity

In the previous code, clients do not contribute equally: some clients perform more local computation (more epochs/steps), while others behave like resource-constrained devices. This is a simplified way of simulating **system and device heterogeneity**.

Complete the following tasks and compare the resulting training dynamics:

1. **Increase the heterogeneity gap**  
   Make one client extremely slow (e.g., `local_epochs=1`, `steps_per_epoch=1`) while keeping the rest as fast clients.

2. **Swap roles**  
   Make the previously “fast” clients slow, and the slow clients fast. Does the global performance change?

3. **Reduce participation**  
   Modify the server strategy so that only a fraction of clients participate each round, for example:
```python
    NUM_CLIENTS = 100
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=0.025,  # Train on 25 clients (each round)
        fraction_evaluate=0.05,  # Evaluate on 50 clients (each round)
        min_fit_clients=20,
        min_evaluate_clients=40,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
        on_fit_config_fn=fit_config
    )
``` 
   How does partial participation interact with heterogeneous client speeds?

4. **Discussion**  
   Based on your observations, explain why system heterogeneity is not only a systems issue but also a learning issue.  
   Which strategy-level mechanisms could mitigate the negative effects of stragglers?

As you run these variations, keep track of:
- aggregated accuracy per round (printed by the server),
- stability across rounds (does accuracy fluctuate?),
- and the number of rounds needed to reach a comparable performance.


In [11]:
import ray
if ray.is_initialized():
    ray.shutdown()

In [5]:
import ray
ray.shutdown()

In [6]:
# INCREASE HETEREOGENITY GAP

import random
class HeterogeneousClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Simulate system heterogeneity via different local computation budgets
        # TODO: change the rules according to the exercise (e.g., make 1 client extremely slow)
        #Hit: You can use self.cid to select a specific client
        if self.cid == 0:
            # Extremely slow client
            local_epochs = 1
            steps_per_epoch = 1
        else:
            # Fast clients
            local_epochs = 5
            steps_per_epoch = 5

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}
        

def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return HeterogeneousClient(cid, model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)


from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]

        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}


from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model and create initial global parameters
    # TODO: create the model and extract initial parameters
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = AggregateCustomMetricStrategy(
        # TODO: set basic FedAvg parameters
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    # Define ServerConfig
    # TODO: set the number of rounds
    config = fl.server.ServerConfig(num_rounds=5)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


from flwr.simulation import run_simulation

NUM_CLIENTS =10
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=23720) 2026-03-18 15:25:17.334673: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(pid=gcs_server) [2026-03-18 15:25:22,738 E 30584 26636] (gcs_server.exe) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


[Server] Round 1 aggregated accuracy: 0.1370


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=13972) WARNING:tensorflow:5 out of the last 35 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000001B599C5C220> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
(ClientAppActor pid=26628) 2026-03-18 15:25:34.137809: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow b

[Server] Round 2 aggregated accuracy: 0.1120


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=3360) WARNING:tensorflow:5 out of the last 35 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000001E628627D80> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fi

[Server] Round 4 aggregated accuracy: 0.1570


(ClientAppActor pid=3360) WARNING:tensorflow:5 out of the last 14 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000001E61A7A80E0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished

[Server] Round 5 aggregated accuracy: 0.1590


(ClientAppActor pid=23720) WARNING :   Manually terminating ClientAppActor
(ClientAppActor pid=3388) WARNING:tensorflow:5 out of the last 35 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x00000213190D5300> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


When the gap between fast and slow clients is increased, the aggregated accuracy becomes lower and more unstable across rounds. From the logs we can see that accuracy fluctuates notably showing no smotth convergence. This happens because clients contribute updates of very different quality. Fast clients produce more informative updates, while slow clients underfit due to limited local training. When aggregated together, these inconsistent updates pull the global model in different directions, slowing convergence.

In [9]:
# SWAP ROLES

import random
class HeterogeneousClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Simulate system heterogeneity via different local computation budgets
        # TODO: change the rules according to the exercise (e.g., make 1 client extremely slow)
        #Hit: You can use self.cid to select a specific client
        if self.cid == 0:
            # Fast client
            local_epochs = 5
            steps_per_epoch = 5
        else:
            # Extremely slow client
            local_epochs = 1
            steps_per_epoch = 1

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}
        

def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return HeterogeneousClient(cid, model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)


from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]

        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}


from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model and create initial global parameters
    # TODO: create the model and extract initial parameters
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = AggregateCustomMetricStrategy(
        # TODO: set basic FedAvg parameters
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    # Define ServerConfig
    # TODO: set the number of rounds
    config = fl.server.ServerConfig(num_rounds=5)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


from flwr.simulation import run_simulation

NUM_CLIENTS =10
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=28040) 2026-03-18 15:27:39.033846: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(pid=gcs_server) [2026-03-18 15:27:43,357 E 24044 31684] (gcs_server.exe) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


[Server] Round 1 aggregated accuracy: 0.0930


(ClientAppActor pid=5032) WARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x0000015097B78FE0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
(ClientAppActor pid=27400) 2026-03-18 15:28:00.174620: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations. [repeated 7x across cluster]
(ClientAppActor p

[Server] Round 2 aggregated accuracy: 0.1060


(ClientAppActor pid=20968) WARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x0000012F1C14F2E0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details. [repeated 3x across cluster]
INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROU

...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.1360


(ClientAppActor pid=17852) WARNING:tensorflow:5 out of the last 35 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000001D21A069E40> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details. [repeated 10x across cluster]
INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SU

[Server] Round 5 aggregated accuracy: 0.1170


(ClientAppActor pid=28040) WARNING :   Manually terminating ClientAppActor
(ClientAppActor pid=17852) WARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000001D2191DD620> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


When the roles are swapped, the global performance does not significantly improve and the instability remains. This shows that the issue is not tied to specific clients, but to the presence of heterogeneity itself.

In [18]:
# REDUCE PARTICIPATION

import random
class HeterogeneousClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Simulate system heterogeneity via different local computation budgets
        # TODO: change the rules according to the exercise (e.g., make 1 client extremely slow)
        #Hit: You can use self.cid to select a specific client
        if random.random()>0.5:
            # "Fast" clients (more compute)
            local_epochs = 5
            steps_per_epoch = 5
        else:
            # "Slow" clients (less compute)
            local_epochs = 3
            steps_per_epoch = 3

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}
        

def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return HeterogeneousClient(cid, model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)

from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]

        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}


from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model and create initial global parameters
    # TODO: create the model and extract initial parameters
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=0.025,  
        fraction_evaluate=0.05,
        min_fit_clients=2,
        min_evaluate_clients=5,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    # Define ServerConfig
    # TODO: set the number of rounds
    config = fl.server.ServerConfig(num_rounds=5)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


from flwr.simulation import run_simulation

NUM_CLIENTS =10
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 10)
(pid=gcs_server) [2026-03-18 15:36:21,860 E 23428 28136] (gcs_server.exe) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(pid=4656) 2026-03-18 15:36:23.834351: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(r

With only a fraction of clients participating per round, performance degrades further. Accuracy remains low, with very limited improvement across rounds. This is because each round is based on very limited and potentially biased data, making updates less representative of the full population.

1. Why is the system heterogeneity also a learning issue?
   
System heterogeneity directly affects how much and how well each client learns locally. This creates:

- Uneven local training: some models are well-trained, others are under-trained.
- Inconsistent updates: gradient points in different directions.
- Bias in aggregation: global model reflects stronger clients more.
    
As a result, the problem impacts the optimization process itself. The global model struggles to converge because it is aggregating updates of unequal quality and representativeness.

2. Which strategy-level mechanisms could mitigate the negative effects of stragglers?

Several federated learning strategies can reduce the negative impact of slow or weak clients:
- Client selection: select only reliable or faster clients per round and avoid including extreme stragglers.
- Adaptive local training: adjust local_epochs and steps_per_epoch per client.
- Weighted aggregation: give more weight to higher-quality updates and reduce the influece of under-trained clients.
- Regularization-based methods (like FedProx): constrain local updates to stay close to the global model.

## From Challenges to Strategies: A Practical Mapping

At this point, we have seen that *challenges* in federated learning are not just abstract limitations: they directly motivate *strategy choices*. A useful way to reason about federated learning is to ask:

**Which challenge is dominant in this scenario, and which strategy design choice addresses it?**

For example, strong non-IID data often motivates strategies that stabilize local training (e.g., FedProx-like ideas), system heterogeneity motivates partial participation or workload control (e.g., limiting steps per epoch), and communication constraints motivate reducing the number of rounds or compressing updates.
